In [0]:
import zipfile, io, os, shutil, glob
import pandas as pd

zip_path = "abfss://data@sand2dbdemodev1data.dfs.core.windows.net/data/bronze/demo/banking/scale_v3_2026_11/after_quantexa_processing/batch-resolver.zip"
output_path = "abfss://data@sand2dbdemodev1data.dfs.core.windows.net/data/bronze/demo/banking/scale_v3_2026_11/after_quantexa_processing/batch-resolver/"

# Read ZIP as binary via Spark
zip_bytes = bytes(spark.read.format("binaryFile").load(zip_path).first()["content"])

# Extract to local temp dir, skipping macOS artifacts
local_tmp = "/tmp/batch-resolver-extract"
if os.path.exists(local_tmp):
    shutil.rmtree(local_tmp)

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
    for member in zf.infolist():
        if member.filename.startswith("__MACOSX") or member.filename.endswith(".DS_Store"):
            continue
        zf.extract(member, local_tmp)

# Read the metadata.delta Parquet files (primary batch-resolver output) into pandas
parquet_dir = os.path.join(local_tmp, "batch-resolver", "metadata.delta")
parquet_files = sorted(glob.glob(os.path.join(parquet_dir, "*.parquet")))
print(f"Found {len(parquet_files)} parquet files in metadata.delta/")

pdf = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)
print(f"Read {len(pdf)} rows, columns: {list(pdf.columns)}")

# Convert complex columns (lists/dicts) to JSON strings to avoid NullType issues
import json
for col in pdf.columns:
    if pdf[col].dtype == "object" and len(pdf) > 0:
        sample = pdf[col].dropna().iloc[0] if pdf[col].notna().any() else None
        if isinstance(sample, (list, dict)):
            pdf[col] = pdf[col].apply(lambda x: json.dumps(x) if x is not None else None)

# Write to ADLS as Delta
from pyspark.sql.types import ArrayType, StringType, NullType
from pyspark.sql import functions as F

sdf = spark.createDataFrame(pdf)

# Fix ArrayType(NullType) columns — cast inner element to StringType
for field in sdf.schema.fields:
    if isinstance(field.dataType, ArrayType) and isinstance(field.dataType.elementType, NullType):
        print(f"Fixing NullType array column: {field.name}")
        sdf = sdf.withColumn(field.name, F.col(field.name).cast(ArrayType(StringType())))

sdf.write.mode("overwrite").option("overwriteSchema", "true").format("delta").save(output_path)

# Clean up
shutil.rmtree(local_tmp)
print(f"Done! {len(pdf)} rows written as Delta to {output_path}")

In [0]:
%sql
-- demo_banking_bronze.scale_v3_2026_11_demo

-- Create Quantexa Banking Demo raw location data
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);


-- risk_location_raw
DROP TABLE IF EXISTS batch_resolver_raw;
CREATE TABLE IF NOT EXISTS batch_resolver_raw
USING DELTA
--OPTIONS  (header = 'true', inferSchema = 'true')
LOCATION 'abfss://data@sand2dbdemodev1data.dfs.core.windows.net/data/bronze/demo/banking/scale_v3_2026_11/after_quantexa_processing/batch-resolver/';


-- https://sand2dbdemodev1data.blob.core.windows.net/data/data/bronze/demo/banking/scale_v3_2026_11/after_quantexa_processing/batch-resolver.zip

In [0]:
%sql
-- demo_banking_bronze.scale_v3_2026_11_demo

-- Create Quantexa Banking Demo raw location data
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);


-- risk_location_raw
DROP TABLE IF EXISTS batch_resolver_small_raw;
CREATE TABLE IF NOT EXISTS batch_resolver_small_raw
USING PARQUET
--OPTIONS  (header = 'true', inferSchema = 'true')
LOCATION 'abfss://data@sand2dbdemodev1data.dfs.core.windows.net/data/bronze/demo/banking/scale_v3_2026_11/batch_resolver_output_sample/part-00000-7daefd8d-58a8-4b7e-a8e7-56673ac843d9-c000.zstd.parquet';


-- https://sand2dbdemodev1data.blob.core.windows.net/data/data/bronze/demo/banking/scale_v3_2026_11/batch_resolver_output_sample/part-00000-7daefd8d-58a8-4b7e-a8e7-56673ac843d9-c000.zstd.parquet